In [4]:
from manim import *
from manim.utils.ipython_magic import ManimMagic

get_ipython().register_magics(ManimMagic)

## 획순 쓰기 함수

`handwritten_text(...)`는 조합된 글자 모양을 유지하면서, 글자 순서(왼→오)와 글자 내부 획 순서(위→아래, 왼→오)를 정렬한 `VGroup`을 반환합니다. 반환값에 `Write(...)`를 적용하면 됩니다.

In [32]:
def handwritten_text(
    content: str,
    font: str = "NanumGothic",
    font_size: float = 72,
    color=WHITE,
    **text_kwargs,
) -> VGroup:
    """조합된 글자 모양을 유지한 채, 글자 순서와 획 순서를 정렬해 반환합니다.

    - 글자 순서: 왼쪽 → 오른쪽
    - 글자 내부 획 순서: 위 → 아래, 왼쪽 → 오른쪽

    사용: self.play(Write(handwritten_text("오늘의 수학")), run_time=4)
    """
    text = Text(content, font=font, font_size=font_size, color=color, **text_kwargs)

    pieces = []
    for i, char in enumerate(text.family_members_with_points()):
        subs = sorted(
            char.get_subpaths(),
            key=lambda p: (-p[:, 1].mean(), p[:, 0].mean()),
        )
        for sp in subs:
            piece = VMobject(color=color).set_points_as_corners(sp)
            piece.char_index = i  # 글자 순서 태그
            pieces.append(piece)

    return VGroup(*pieces)

## 19. 함수 사용 예시 — 판서 스타일

위의 `handwritten_text` 함수를 이용해 칠판 배경에 씁니다.

In [40]:
%%manim -ql -v WARNING HandwrittenExample
class HandwrittenExample(Scene):
    def construct(self):
        self.camera.background_color = "#1d2b21"
        text = handwritten_text("오늘의 수학", font_size=72)
        chalk = Line(text.get_corner(DL), text.get_corner(DR), color=WHITE)
        chalk.next_to(text, DOWN)
        group = VGroup(text, chalk)

        self.play(Write(text), run_time=1)
        self.play(Create(chalk))
        self.wait(0.3)
        # 왼쪽 위로 옮기면서 작아지기
        self.play(group.animate.scale(0.3).to_corner(UL), run_time=1)
        self.wait(0.5)

Manim Community v0.21.0

## PDF → Manim 변환 함수 (원본 픽셀 그대로 크롭)

이 문제 PDF는 수식 기호가 유니코드 텍스트가 아니라 커스텀 폰트(Private Use Area 코드)로 인코딩되어 있어, 텍스트를 문자로 재해석하면 깨집니다(□같은 대체 글자로 표시됨). 그래서 텍스트 **줄 단위 영역**과 **삽입 이미지**, **텍스트에 속하지 않는 도형(그림) 영역**을 원본 그대로 고해상도로 크롭해 `ImageMobject`로 재구성합니다. 폰트 재해석이 없어 원본과 픽셀 단위로 동일하게 보입니다.

In [57]:
def pdf_page_to_manim_group(
    pdf_path: str,
    page_number: int = 0,
    region: tuple[float, float, float, float] | None = None,
    target_width: float | None = None,
    target_height: float | None = None,
    dpi: int = 300,
    line_padding: float = 0.15,
) -> Group:
    """PDF 페이지를 텍스트 줄/삽입 이미지/도형 단위로 크롭해 원본 그대로 재구성합니다.

    글자를 유니코드로 재해석하지 않고 픽셀 그대로 크롭하기 때문에, 커스텀 인코딩
    폰트나 수식 기호도 원본과 동일하게 보입니다. 배경은 투명(alpha)하게 크롭되어
    각 줄이 사각형이 아니라 글자 모양 그대로 보입니다. 각 조각은 개별 ImageMobject라서
    Write 대신 FadeIn 등으로 하나씩 등장시킬 수 있습니다.
    폭/높이 중 더 제약적인 쪽에 맞춰 스케일하므로 세로로 긴 페이지도 화면 안에 전부 들어옵니다.

    - region: (x0, y0, x1, y1) PDF 좌표(포인트 단위)로 페이지 일부(예: 문제 하나)만 추출.
      None이면 페이지 전체를 사용.

    사용: self.add(pdf_page_to_manim_group("문제.pdf", 0, region=(75, 595, 415, 720)))
    """
    doc = pymupdf.open(pdf_path)
    page = doc[page_number]
    bounds = pymupdf.Rect(region) if region is not None else page.rect
    area_w, area_h = bounds.width, bounds.height
    if target_width is None:
        target_width = config.frame_width - 1
    if target_height is None:
        target_height = config.frame_height - 0.5
    scale = min(target_width / area_w, target_height / area_h)
    zoom = dpi / 72

    def to_manim_xy(px: float, py: float) -> tuple[float, float]:
        cx, cy = bounds.x0 + area_w / 2, bounds.y0 + area_h / 2
        return (px - cx) * scale, (cy - py) * scale

    def crop_to_image_mobject(rect, tag: str) -> ImageMobject:
        # alpha=True: 배경(종이 흰색)을 투명하게 남겨 글자 모양만 남김
        pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), clip=rect, alpha=True)
        img_path = f"pdf_crop_{page_number}_{tag}.png"
        pix.save(img_path)
        mobj = ImageMobject(img_path)
        mobj.scale_to_fit_width(max(rect.width * scale, 0.01))
        cx, cy = to_manim_xy((rect.x0 + rect.x1) / 2, (rect.y0 + rect.y1) / 2)
        mobj.move_to([cx, cy, 0])
        return mobj

    group = Group()
    text_rects = []

    # 1) 텍스트 줄: 여백을 조금 더해 분수선/근호처럼 줄 밖으로 튀어나오는 기호까지 포함
    for block in page.get_text("dict")["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            if not "".join(s["text"] for s in line["spans"]).strip():
                continue
            x0, y0, x1, y1 = line["bbox"]
            pad_y = (y1 - y0) * line_padding + 1
            rect = pymupdf.Rect(x0 - 1, y0 - pad_y, x1 + 1, y1 + pad_y) & page.rect
            if region is not None:
                rect &= bounds
                if rect.is_empty:
                    continue
            group.add(crop_to_image_mobject(rect, f"line{len(text_rects)}"))
            text_rects.append(rect)

    # 2) 삽입된 래스터 이미지(사진, 그래프 등) - 원본이 사진이므로 배경 유지
    for i, img in enumerate(page.get_images(full=True)):
        bbox = page.get_image_bbox(img)
        if region is not None:
            bbox &= bounds
            if bbox.is_empty:
                continue
        group.add(crop_to_image_mobject(bbox, f"img{i}"))

    # 3) 텍스트 줄에 속하지 않는 벡터 도형(기하 그림 등) - 흩어진 선/곡선을 하나의 그림으로 묶어 크롭
    stray = [
        p["rect"] for p in page.get_drawings()
        if not p["rect"].is_empty and not any(_overlap_ratio(p["rect"], t) > 0.5 for t in text_rects)
    ]
    for i, cluster in enumerate(_merge_rects(stray)):
        if region is not None:
            cluster &= bounds
        if cluster.width > 2 and cluster.height > 2:
            group.add(crop_to_image_mobject(cluster, f"fig{i}"))

    doc.close()
    return group

In [59]:
def pdf_page_to_manim_group_chars(
    pdf_path: str,
    page_number: int = 0,
    region: tuple[float, float, float, float] | None = None,
    target_width: float | None = None,
    target_height: float | None = None,
    dpi: int = 300,
    char_padding: float = 0.15,
) -> Group:
    """pdf_page_to_manim_group의 글자 단위 버전. 줄이 아니라 글자 하나하나를 개별
    ImageMobject로 크롭해서, 낱글자 단위로 Write/FadeIn 등 애니메이션을 걸 수 있습니다.

    사용: self.play(FadeIn(m) for m in pdf_page_to_manim_group_chars("문제.pdf", 0))
    """
    doc = pymupdf.open(pdf_path)
    page = doc[page_number]
    bounds = pymupdf.Rect(region) if region is not None else page.rect
    area_w, area_h = bounds.width, bounds.height
    if target_width is None:
        target_width = config.frame_width - 1
    if target_height is None:
        target_height = config.frame_height - 0.5
    scale = min(target_width / area_w, target_height / area_h)
    zoom = dpi / 72

    def to_manim_xy(px: float, py: float) -> tuple[float, float]:
        cx, cy = bounds.x0 + area_w / 2, bounds.y0 + area_h / 2
        return (px - cx) * scale, (cy - py) * scale

    def crop_to_image_mobject(rect, tag: str) -> ImageMobject:
        pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), clip=rect, alpha=True)
        img_path = f"pdf_crop_{page_number}_{tag}.png"
        pix.save(img_path)
        mobj = ImageMobject(img_path)
        mobj.scale_to_fit_width(max(rect.width * scale, 0.01))
        cx, cy = to_manim_xy((rect.x0 + rect.x1) / 2, (rect.y0 + rect.y1) / 2)
        mobj.move_to([cx, cy, 0])
        return mobj

    group = Group()
    char_rects = []

    # 1) 글자 하나하나 크롭 (rawdict은 span 안에 chars 리스트로 낱글자 bbox를 제공)
    for block in page.get_text("rawdict")["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                for ch in span["chars"]:
                    if not ch["c"].strip():
                        continue
                    x0, y0, x1, y1 = ch["bbox"]
                    pad_y = (y1 - y0) * char_padding + 1
                    rect = pymupdf.Rect(x0 - 1, y0 - pad_y, x1 + 1, y1 + pad_y) & page.rect
                    if region is not None:
                        rect &= bounds
                        if rect.is_empty:
                            continue
                    group.add(crop_to_image_mobject(rect, f"char{len(char_rects)}"))
                    char_rects.append(rect)

    # 2) 삽입된 래스터 이미지(사진, 그래프 등)
    for i, img in enumerate(page.get_images(full=True)):
        bbox = page.get_image_bbox(img)
        if region is not None:
            bbox &= bounds
            if bbox.is_empty:
                continue
        group.add(crop_to_image_mobject(bbox, f"img{i}"))

    # 3) 글자에 속하지 않는 벡터 도형(기하 그림 등)
    stray = [
        p["rect"] for p in page.get_drawings()
        if not p["rect"].is_empty and not any(_overlap_ratio(p["rect"], t) > 0.5 for t in char_rects)
    ]
    for i, cluster in enumerate(_merge_rects(stray)):
        if region is not None:
            cluster &= bounds
        if cluster.width > 2 and cluster.height > 2:
            group.add(crop_to_image_mobject(cluster, f"fig{i}"))

    doc.close()
    return group

In [67]:
def _has_pua(text: str) -> bool:
    """이 PDF는 수식(HyhwpEQ 폰트)만 유니코드 Private Use Area 코드로 인코딩되어 있음."""
    return any(0xE000 <= ord(c) <= 0xF8FF for c in text)


def pdf_page_to_manim_text_group(
    pdf_path: str,
    page_number: int = 0,
    region: tuple[float, float, float, float] | None = None,
    target_width: float | None = None,
    target_height: float | None = None,
    dpi: int = 300,
    default_font: str = "NanumGothic",
) -> Group:
    """일반 한글/영문/숫자는 진짜 Text mobject로, 수식(PUA 인코딩 글리프)만 이미지로
    변환하는 하이브리드 버전. 글자 단위(span이 아니라 char)로 처리해야 대체 폰트의
    자간 차이로 인한 겹침을 피할 수 있습니다.

    사용: self.play(Write(pdf_page_to_manim_text_group("문제.pdf", 0)))
    """
    doc = pymupdf.open(pdf_path)
    page = doc[page_number]
    bounds = pymupdf.Rect(region) if region is not None else page.rect
    area_w, area_h = bounds.width, bounds.height
    if target_width is None:
        target_width = config.frame_width - 1
    if target_height is None:
        target_height = config.frame_height - 0.5
    scale = min(target_width / area_w, target_height / area_h)
    zoom = dpi / 72

    def to_manim_xy(px: float, py: float) -> tuple[float, float]:
        cx, cy = bounds.x0 + area_w / 2, bounds.y0 + area_h / 2
        return (px - cx) * scale, (cy - py) * scale

    def crop_to_image_mobject(rect, tag: str) -> ImageMobject:
        pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), clip=rect, alpha=True)
        img_path = f"pdf_crop_{page_number}_{tag}.png"
        pix.save(img_path)
        mobj = ImageMobject(img_path)
        mobj.scale_to_fit_width(max(rect.width * scale, 0.01))
        cx, cy = to_manim_xy((rect.x0 + rect.x1) / 2, (rect.y0 + rect.y1) / 2)
        mobj.move_to([cx, cy, 0])
        return mobj

    group = Group()
    covered_rects = []
    tag = 0

    for block in page.get_text("rawdict")["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                color_int = span["color"]
                color = "#%02x%02x%02x" % ((color_int >> 16) & 255, (color_int >> 8) & 255, color_int & 255)
                for ch in span["chars"]:
                    c = ch["c"]
                    if not c.strip():
                        continue
                    x0, y0, x1, y1 = ch["bbox"]
                    rect = pymupdf.Rect(x0, y0, x1, y1) & page.rect
                    if region is not None:
                        rect &= bounds
                        if rect.is_empty:
                            continue

                    if _has_pua(c):
                        pad_y = (y1 - y0) * 0.15 + 1
                        padded = pymupdf.Rect(rect.x0 - 1, rect.y0 - pad_y, rect.x1 + 1, rect.y1 + pad_y) & page.rect
                        group.add(crop_to_image_mobject(padded, f"eq{tag}"))
                    else:
                        mobj = Text(c, font=default_font, color=color)
                        # 대체 폰트의 자간이 원본과 달라서, 글자 단위로 원본 칸에 맞춰 늘여야 안 겹침
                        mobj.stretch_to_fit_width(max(rect.width * scale, 0.01))
                        mobj.stretch_to_fit_height(max(rect.height * scale, 0.01))
                        cx, cy = to_manim_xy((rect.x0 + rect.x1) / 2, (rect.y0 + rect.y1) / 2)
                        mobj.move_to([cx, cy, 0])
                        group.add(mobj)

                    covered_rects.append(rect)
                    tag += 1

    for i, img in enumerate(page.get_images(full=True)):
        bbox = page.get_image_bbox(img)
        if region is not None:
            bbox &= bounds
            if bbox.is_empty:
                continue
        group.add(crop_to_image_mobject(bbox, f"img{i}"))

    stray = [
        p["rect"] for p in page.get_drawings()
        if not p["rect"].is_empty and not any(_overlap_ratio(p["rect"], t) > 0.5 for t in covered_rects)
    ]
    for i, cluster in enumerate(_merge_rects(stray)):
        if region is not None:
            cluster &= bounds
        if cluster.width > 2 and cluster.height > 2:
            group.add(crop_to_image_mobject(cluster, f"fig{i}"))

    doc.close()
    return group

## 20. 사용 예시 — 2번 문제만 재현

`region`으로 페이지에서 2번 문제 영역만 잘라 화면에 꽉 차게 재현합니다.

In [58]:
%%manim -ql -v WARNING PdfProblem2Example
class PdfProblem2Example(Scene):
    def construct(self):
        self.camera.background_color = "#264653"  # 투명 배경 확인용 배경색
        problem2 = pdf_page_to_manim_group(
            "mathB_1_mun_ZM4M1F58.pdf",
            page_number=0,
            region=(75, 595, 415, 720),  # 2번 문제 영역 (PDF 좌표)
        )
        problem2.move_to(ORIGIN)

        self.play(FadeIn(problem2))
        self.wait(1)

Manim Community v0.21.0

In [61]:
# 2번 문제 영역의 span별 폰트/텍스트 확인 -> 정상 유니코드 vs PUA(수식 폰트) 구분
doc = pymupdf.open("mathB_1_mun_ZM4M1F58.pdf")
page = doc[0]
region = pymupdf.Rect(75, 595, 415, 720)
for block in page.get_text("dict")["blocks"]:
    if block["type"] != 0:
        continue
    for line in block["lines"]:
        if not (pymupdf.Rect(line["bbox"]) & region):
            continue
        for span in line["spans"]:
            is_pua = any(0xE000 <= ord(c) <= 0xF8FF for c in span["text"])
            print("PUA" if is_pua else "OK ", span["font"], repr(span["text"]))
doc.close()

OK  *½Å¸í-°ß¸íÁ¶ '1'
OK  *½Å¸í-°ß¸íÁ¶ '1'
OK  *½Å¸í-°ß¸íÁ¶ '20'
OK  *#ÅÂ°íµñ '5'
OK  *#ÅÂ°íµñ '지선다형'
OK  *ÇÑ¾ç°ß¸íÁ¶ '1.'
OK  HyhwpEQ ' '
PUA HyhwpEQ '\ue035'
PUA HyhwpEQ '\ue06d'
PUA HyhwpEQ '\ue035'
PUA HyhwpEQ '\ue036'
PUA HyhwpEQ '× \ue037'
PUA HyhwpEQ '\ue046'
PUA HyhwpEQ '\ue06d'
PUA HyhwpEQ '\ue035'
PUA HyhwpEQ '\ue034'
OK  *½Å¸í-Áß¸íÁ¶ '의 값은'
OK  *ÇÑ¾ç½Å¸íÁ¶ '? [2'
OK  *½Å¸í-Áß¸íÁ¶ '점'
OK  *ÇÑ¾ç½Å¸íÁ¶ ']'
OK  *ÇÑ¾ç½Å¸íÁ¶ '①'
PUA HyhwpEQ '\ue035'
PUA HyhwpEQ '\ue046\ue034'
OK  *ÇÑ¾ç½Å¸íÁ¶ '②'
PUA HyhwpEQ '\ue035'
PUA HyhwpEQ '\ue046'
PUA HyhwpEQ '\ue06d'
PUA HyhwpEQ '\ue035'
PUA HyhwpEQ '\ue034'
OK  *ÇÑ¾ç½Å¸íÁ¶ '③'
PUA HyhwpEQ '\ue034'
OK  *ÇÑ¾ç½Å¸íÁ¶ '④'
PUA HyhwpEQ '\ue035'
PUA HyhwpEQ '\ue06d'
PUA HyhwpEQ '\ue035'
PUA HyhwpEQ '\ue034'
OK  *ÇÑ¾ç½Å¸íÁ¶ '⑤'
PUA HyhwpEQ '\ue035'
OK  *ÇÑ¾ç°ß¸íÁ¶ '2.'
OK  *½Å¸í-Áß¸íÁ¶ ' '
OK  *½Å¸í-Áß¸íÁ¶ '함수 '
PUA HyhwpEQ '\ue0ea'
PUA HyhwpEQ '\ue044'
PUA HyhwpEQ '\ue0fc'
PUA HyhwpEQ '\ue045'
PUA HyhwpEQ '\ue047'
PUA HyhwpEQ '\ue035\ue0fc'
PUA Hyh

## 21. 사용 예시 — 글자 단위 추출로 2번 문제 재현

`pdf_page_to_manim_group_chars`는 낱글자 단위로 크롭하므로, 글자가 하나씩 나타나는 연출(`lag_ratio`)이 가능합니다.

In [60]:
%%manim -ql -v WARNING PdfProblem2CharsExample
class PdfProblem2CharsExample(Scene):
    def construct(self):
        self.camera.background_color = "#264653"
        problem2 = pdf_page_to_manim_group_chars(
            "mathB_1_mun_ZM4M1F58.pdf",
            page_number=0,
            region=(75, 595, 415, 720),
        )
        problem2.move_to(ORIGIN)

        self.play(FadeIn(problem2, lag_ratio=0.02), run_time=3)
        self.wait(1)

Manim Community v0.21.0

## 22. 문제 번호 기반 자동 크롭

`auto_problem_region(...)`은 PDF에서 문제 번호를 찾고, 다음 문제 번호가 나오기 전까지의 텍스트와 도형을 자동으로 묶어 크롭 영역을 계산합니다. `problem_number`만 바꾸면 같은 페이지의 다른 문제도 재사용할 수 있습니다.

In [22]:
import re
import pymupdf


def auto_problem_region(
    pdf_path: str,
    page_number: int = 0,
    problem_number: int = 2,
    padding_x: float = 10,
    padding_y: float = 20,
    paragraph_gap: float = 80,
) -> pymupdf.Rect:
    """문제 번호, 2단 편집 열, 문단 간격을 기준으로 문제 영역을 자동 반환합니다."""
    doc = pymupdf.open(pdf_path)
    page = doc[page_number]
    lines = []

    for block in page.get_text("dict")["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            text = "".join(span["text"] for span in line["spans"])
            lines.append((pymupdf.Rect(line["bbox"]), text))

    problem_lines = []
    for rect, text in lines:
        match = re.match(r"\s*(\d+)\.", text)
        if match:
            problem_lines.append((int(match.group(1)), rect))

    matches = [rect for number, rect in problem_lines if number == problem_number]
    if not matches:
        doc.close()
        raise ValueError(f"문제 번호 {problem_number}번을 찾지 못했습니다.")

    start_rect = matches[0]
    vertical_boundaries = []
    for drawing in page.get_drawings():
        rect = drawing["rect"]
        if rect.width <= 5 and rect.height > page.rect.height * 0.5:
            vertical_boundaries.append(rect.x0)

    left_boundary = max([x for x in vertical_boundaries if x < start_rect.x0], default=page.rect.x0)
    right_boundary = min([x for x in vertical_boundaries if x > start_rect.x1], default=page.rect.x1)
    column_band = pymupdf.Rect(left_boundary, page.rect.y0, right_boundary, page.rect.y1)

    next_starts = [
        rect.y0
        for number, rect in problem_lines
        if rect.y0 > start_rect.y1 + 10
        and number != problem_number
        and rect.intersects(column_band)
    ]
    next_y = min(next_starts, default=page.rect.y1)
    column_lines = sorted(
        [
            rect
            for rect, _ in lines
            if rect.y1 >= start_rect.y0
            and rect.y0 < next_y
            and rect.intersects(column_band)
        ],
        key=lambda rect: (rect.y0, rect.x0),
    )

    selected_lines = []
    previous_y1 = start_rect.y0
    for rect in column_lines:
        if selected_lines and rect.y0 - previous_y1 > paragraph_gap:
            break
        selected_lines.append(rect)
        previous_y1 = max(previous_y1, rect.y1)

    if not selected_lines:
        doc.close()
        raise ValueError(f"문제 {problem_number}번의 내용을 찾지 못했습니다.")

    band = pymupdf.Rect(left_boundary, start_rect.y0, right_boundary, max(rect.y1 for rect in selected_lines))
    content = list(selected_lines)
    for drawing in page.get_drawings():
        original = drawing["rect"]
        if original.height > band.height * 0.8 or original.width > page.rect.width * 0.8:
            continue
        clipped = original & band
        if not clipped.is_empty:
            content.append(clipped)

    result = pymupdf.Rect(
        min(rect.x0 for rect in content) - padding_x,
        min(rect.y0 for rect in content) - padding_y,
        max(rect.x1 for rect in content) + padding_x,
        max(rect.y1 for rect in content) + padding_y,
    ) & page.rect
    doc.close()
    return result

In [28]:
%%manim -ql -v WARNING PdfProblem2AutoCropExample
class PdfProblem2AutoCropExample(Scene):
    def construct(self):
        self.camera.background_color = WHITE
        pdf_path = "mathB_1_mun_ZM4M1F58.pdf"
        problem_number = 2
        region = auto_problem_region(
            pdf_path,
            page_number=0,
            problem_number=problem_number,
        )

        doc = pymupdf.open(pdf_path)
        pixmap = doc[0].get_pixmap(
            matrix=pymupdf.Matrix(3, 3),
            clip=region,
            alpha=False,
        )
        image_path = f"pdf_problem_{problem_number}.png"
        pixmap.save(image_path)
        doc.close()

        problem = ImageMobject(image_path)
        problem.scale_to_fit_width(config.frame_width - 1)
        problem.move_to(ORIGIN)
        self.play(FadeIn(problem), run_time=1.5)
        self.wait(1)

Manim Community v0.21.0